# scWAT Xenium slide-level QC summary

This notebook is the sole reader-facing QC report for the four independently processed scWAT sections. Sections are technical units; the two mice are the biological units.

## Setup

### Parameters

Inputs: four completed section bundles below `RUN_ROOT`, the verified sample manifest, and versioned QC settings. Outputs: combined tables and Cell-style figures below `${RUN_ROOT}/slide_summary/`.

In [ ]:
PROJECT_ROOT <- "/dssg/home/acct-svetoslav_chakarov/svetoslav_chakarov/Lab_members/Yanan_Hu/YNH_Xenium"
PIPELINE_REPO <- file.path(PROJECT_ROOT, "adipose_analysis", "YNH_Xenium_scWAT")
RUN_LABEL <- "full_notebook_qc_v2"
EXPECTED_SECTION_COUNT <- 4L
METADATA_PATH <- file.path(PIPELINE_REPO, "config", "scwat_sample_manifest.tsv")
EXTENDED_QC_CONFIG_PATH <- file.path(PIPELINE_REPO, "config", "extended_qc_defaults.tsv")
SUBSET_REFERENCE_PATH <- file.path(PIPELINE_REPO, "config", "subset_qc_reference.tsv")


In [ ]:
RUN_ROOT <- file.path(PROJECT_ROOT, "adipose_analysis", "scwat_qc_outputs", RUN_LABEL)
source(file.path(PIPELINE_REPO, "R", "source.R"))
for (package in c("Matrix", "jsonlite", "ggplot2")) require_package(package)
assert_path_within(PROJECT_ROOT, RUN_ROOT)
assert_path_within(PROJECT_ROOT, tempdir())
stopifnot(EXPECTED_SECTION_COUNT == 4L)
cat("Slide QC run root:", RUN_ROOT, "\n")


## Inputs and validation

Exactly four independently completed core and extended section bundles are required. The verified manifest maps 62308/62309 to Mouse 1 and 62310/62311 to Mouse 2. Left/right is not an analysis factor.

In [ ]:
coverage <- validate_four_section_outputs(RUN_ROOT, paste0("Region_", seq_len(EXPECTED_SECTION_COUNT)))
extended_coverage <- validate_four_extended_section_outputs(RUN_ROOT, coverage$region_id)
stopifnot(identical(coverage$region_id, extended_coverage$region_id))
slide_data <- read_slide_qc_outputs(RUN_ROOT, coverage$region_id)
slide_summary <- summarise_slide_qc(slide_data)
extended_slide_data <- read_extended_slide_qc_outputs(RUN_ROOT, coverage$region_id)
manifest <- utils::read.delim(METADATA_PATH, check.names = FALSE)
extended_config <- read_extended_qc_config(EXTENDED_QC_CONFIG_PATH)
subset_reference <- utils::read.delim(SUBSET_REFERENCE_PATH, check.names = FALSE)
extended_slide_summary <- summarise_extended_slide_qc(extended_slide_data, slide_summary$section_summary, manifest, extended_config, subset_reference)
stopifnot(length(unique(slide_data$cell_metadata$region_id)) == 4L)
list(core = coverage, extended = extended_coverage, cells = nrow(slide_data$cell_metadata))


## TL;DR and QC decision

The table below is the review entry point. Direct poor-cycle alarms are available for Regions 1, 2, and 4. Exact cycle/channel/codeword identity is `REQUIRES_10X_DIAGNOSTICS`. Region 3 requires morphology review; Region 4 additionally requires segmentation/cell-area review.

In [ ]:
qc_decision <- merge(slide_summary$section_summary, slide_summary$readiness[, c("region_id", "status")], by = "region_id", all.x = TRUE, sort = FALSE)
qc_decision <- qc_decision[match(coverage$region_id, qc_decision$region_id), , drop = FALSE]
direct_alarm_regions <- unique(extended_slide_data$cycle_alarm_evidence$region_id[extended_slide_data$cycle_alarm_evidence$evidence_status == "DIRECT_EVIDENCE"])
qc_decision$direct_poor_cycle_alarm <- qc_decision$region_id %in% direct_alarm_regions
qc_decision$reported_qc_status <- ifelse(qc_decision$region_id == "Region_3" & qc_decision$status == "PASS", "CONDITIONAL_PASS", qc_decision$status)
qc_decision$required_next_action <- c("Obtain 10x poor-cycle diagnostics", "Obtain 10x poor-cycle diagnostics", "Review morphology in FDR-positive hotspot bins", "Obtain 10x diagnostics and review segmentation/cell area")[match(qc_decision$region_id, paste0("Region_", 1:4))]
qc_decision[, c("region_id", "input_cells", "core_pass_fraction", "review_fraction", "direct_poor_cycle_alarm", "reported_qc_status", "required_next_action")]


## Core QC distributions

These are descriptive section-level and cell-level QC summaries. No cells are automatically deleted, and cell-level distributions do not create biological replication.

In [ ]:
slide_summary$section_summary
stopifnot(all(slide_summary$section_summary$cells_deleted == 0L))
slide_plots <- plot_slide_qc(slide_data, slide_summary)
for (plot in slide_plots) print(plot)


## Alarm evidence and candidate genes

The alarm table reports directly available evidence. The candidate list uses cross-section transcript depletion/quality patterns and remains `CANDIDATE_NOT_CONFIRMED`; it is not a definitive cycle-to-gene map. Exact cycle identity is `REQUIRES_10X_DIAGNOSTICS` and must come from 10x diagnostic output.

In [ ]:
extended_slide_data$cycle_alarm_evidence
candidate_display <- extended_slide_summary$candidates[extended_slide_summary$candidates$section_candidate_flag, , drop = FALSE]
gene_sets <- unique(extended_slide_data$gene_quality[, c("gene", "gene_set")])
candidate_display <- merge(candidate_display, gene_sets, by = "gene", all.x = TRUE, sort = FALSE)
candidate_counts <- if (nrow(candidate_display)) aggregate(section_candidate_flag ~ region_id + section_evidence_status, candidate_display, sum) else data.frame(region_id = character(), section_evidence_status = character(), section_candidate_flag = integer())
candidate_counts
if (nrow(candidate_display)) with(candidate_display, table(region_id, gene_set, useNA = "ifany"))
candidate_display <- candidate_display[order(candidate_display$region_id, candidate_display$evidence_tier, candidate_display$log2_count_ratio, candidate_display$q20_difference), , drop = FALSE]
candidate_display[seq_len(min(30L, nrow(candidate_display))), , drop = FALSE]


## Subset versus full-data burden

The comparison is descriptive across four technical sections. `NOT_RUN_LOCAL_SUBSET` means the full-data ranking remains an HPC checkpoint; rank correlations across only four sections are not biological evidence.

In [ ]:
extended_slide_summary$ranking
extended_slide_summary$rank_agreement


## Spatial QC

Global kNN clustering, tissue-edge proxies, dense-cell proxies, and candidate hotspot bins are coordinate-based diagnostics. A hotspot is only `MORPHOLOGY_REVIEW_REQUIRED`; folds, tears, tissue edges, and aggregates require image review.

In [ ]:
extended_slide_data$spatial_global
extended_slide_data$spatial_edge_density
extended_slide_data$manual_review_manifest


## Within-mouse concordance

The verified technical pairs are 62308/62309 for Mouse 1 and 62310/62311 for Mouse 2. Thresholds are advisory. The Region 3/4 cell-area contrast is reviewed separately because it was not part of the original concordance gate.

In [ ]:
extended_slide_summary$concordance$summary


## Diagnostic Cell-style figures

Colors and scales are consistent across sections where scientifically appropriate. These plots support review rather than biological inference.

In [ ]:
extended_slide_plots <- plot_extended_slide_qc(extended_slide_data, extended_slide_summary)
for (plot in extended_slide_plots) print(plot)


## Final QC decision and next actions

Regions 1 and 2 remain `HOLD` pending 10x diagnostics. Region 3 is a conditional pass pending morphology review. Region 4 remains `HOLD` pending 10x diagnostics plus segmentation/cell-area review. Comparative gene-level analysis remains on hold until the affected-gene decision is frozen.

In [ ]:
qc_decision[, c("region_id", "reported_qc_status", "required_next_action")]
extended_slide_summary$status
cat("Overall slide QC status:", slide_summary$overall_status, "\n")


## Outputs and reload checks

All tables, figures, and serialized objects are written below `${RUN_ROOT}/slide_summary/`. The notebook then reloads and validates the complete artifact contract.

In [ ]:
slide_artifacts <- write_slide_qc_artifacts(PROJECT_ROOT, RUN_ROOT, slide_data, slide_summary, slide_plots)
extended_slide_artifacts <- write_extended_slide_qc_artifacts(PROJECT_ROOT, RUN_ROOT, extended_slide_data, extended_slide_summary, extended_slide_plots)
saved_summary <- readRDS(file.path(RUN_ROOT, "slide_summary", "slide_qc_summary.rds"))
stopifnot(nrow(saved_summary$data$coverage) == 4L)
stopifnot(length(unique(saved_summary$data$cell_metadata$region_id)) == 4L)
stopifnot(validate_extended_slide_qc_artifacts(RUN_ROOT, stop_on_error = TRUE))
list(core = data.frame(artifact = basename(slide_artifacts), path = slide_artifacts),
     extended = data.frame(artifact = basename(extended_slide_artifacts), path = extended_slide_artifacts))
